# 04 · Score scale: calibration across cell types and recall-based thresholds

CisLink scores are probabilities on the scale of the K562 CRISPR training pairs. The calibration slope (logistic regression of the label on logit(score); 1 = scale preserved) measures whether that scale transfers to other cell types; the recall-based thresholds give a scale-free way to read scores.

In [1]:
import pandas as pd

from cislink import datasets, load_model
from cislink import evaluate as ev

P = {c: datasets.crispr_pairs(c) for c in ("K562", "HCT116", "GM12878")}   # cells with both models (WTC11: CisLink-Base only)
rows = []
for c, p in P.items():
    for m in ("CisLink-Extended", "CisLink-Base"):
        r = ev.calibration(p.label, p[m]); r.update(set=c, model=m); rows.append(r)
pd.DataFrame(rows)[["set", "model", "slope", "intercept", "mean_score", "positive_rate"]].round(3)

,set,model,slope,intercept,mean_score,positive_rate
0,K562,CisLink-Extended,0.873,0.023,0.054,0.061
1,K562,CisLink-Base,0.866,0.003,0.053,0.061
2,HCT116,CisLink-Extended,1.125,-0.922,0.136,0.091
3,HCT116,CisLink-Base,1.076,-0.261,0.113,0.091
4,GM12878,CisLink-Extended,1.093,0.875,0.188,0.235
5,GM12878,CisLink-Base,1.343,0.674,0.202,0.235


In [2]:
k = P["K562"]   # held-out K562: the thresholds were set on cross-validated training predictions, so this is a test
for m in ("extended", "base"):
    model = load_model(m)
    print(model.name, "released:", model.thresholds["recall_70"], "(70%)", model.thresholds["recall_50"], "(50%);",
          "held-out K562 scores at the same recall:", {r: round(ev.recall_threshold(k.label, k[model.name], r / 100), 3) for r in (70, 50)})

CisLink-Extended released: 0.269 (70%) 0.565 (50%); held-out K562 scores at the same recall: {70: 0.195, 50: 0.493}
CisLink-Base released: 0.23 (70%) 0.476 (50%); held-out K562 scores at the same recall: {70: 0.147, 50: 0.402}


In [3]:
t = []
for c, p in P.items():
    for m in ("extended", "base"):
        model = load_model(m); hit = p[model.name] >= model.threshold(70)
        t.append({"set": c, "model": model.name, "threshold": model.threshold(70), "recall": p.label[hit].sum() / p.label.sum(),
                  "precision": p.label[hit].mean(), "positive rate": p.label.mean()})
pd.DataFrame(t).round(3)

,set,model,threshold,recall,precision,positive rate
0,K562,CisLink-Extended,0.269,0.643,0.632,0.061
1,K562,CisLink-Base,0.230,0.600,0.511,0.061
2,HCT116,CisLink-Extended,0.269,0.882,0.484,0.091
3,HCT116,CisLink-Base,0.230,0.794,0.458,0.091
4,GM12878,CisLink-Extended,0.269,0.812,0.765,0.235
5,GM12878,CisLink-Base,0.230,0.938,0.714,0.235


In the manuscript the scale transfers to HCT116 and GM12878 (CisLink-Extended slopes 1.12 and 1.09) but scores are overconfident in osteoblasts (0.56–0.64); the 70%-recall threshold keeps 49–94% recall in other cell types.